# Phase 5 LSI with c-3 (hourglass encoder-decoder): LoRA TEMPLATE, with a frozen-layer cache

**Setup:** Add Input -> Your Work -> phase-1-c-3. Accelerator GPU T4, Internet on.

Run order: Cell 0 -> A -> B -> C -> D -> E0 -> E2. To run E2 again, re-run Cell A and E0 first.

**Plain LoRA is not on the Phase 5 technique list** (QLoRA is): it is a valid submission but not one of the required distinct techniques.

**Why it is fast:** adapters go only into the stages from `LORA_FROM` upwards; the stages below are run once over every train/dev window and cached, so each epoch runs only the adapted stages + head.

Yours in E2: `LoRALinear` (raises NotImplementedError) and the LoRA values (`LORA_FROM`, `LORA_RANK`, `LORA_ALPHA`, `LORA_TARGETS`, `LORA_DROPOUT`, `LR`, `HEAD_LR`, `EPOCHS`). Data, loss and threshold settings are pre-set to your c-6 Phase 5 values; the head is your c-6 `LSIHead`.

## Cell 0: classes from your Phase 1 c-3 notebook (cells 1, 8, 20, 22, 23, in that order)

In [1]:
# --- Phase 1 c-3 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict

In [2]:
# --- Phase 1 c-3 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [3]:
# --- Phase 1 c-3 notebook, cell 20 (verbatim) ---
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# --- Phase 1 c-3 notebook, cell 22 (verbatim) ---
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    shorten_factor: int = 4
    n_pre_layers: int = 1
    n_bottleneck_layers: int = 4
    n_post_layers: int = 1

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x

# ---------------------------------------------------------
# 3. Hierarchical Resampling Operations
# ---------------------------------------------------------
class Downsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        B, L, D = x.shape
        x = x.view(B, L // self.factor, self.factor, D)
        return x.mean(dim=2)


class Upsample(nn.Module):
    def __init__(self, factor: int):
        super().__init__()
        self.factor = factor

    def forward(self, x: torch.Tensor):
        return x.repeat_interleave(self.factor, dim=1)


def shift_right(x: torch.Tensor, shift: int):
    if shift == 0:
        return x
    shifted = F.pad(x, (0, 0, shift, 0), value=0.0)
    return shifted[:, :-shift, :]

# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class HourglassEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        
        # Positional Encoding (using standard learned embedding for simplicity, 
        # replace with sinusoidal if strictly required)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        # ENCODER
        self.encoder_pre = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.encoder_hourglass = EncoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.encoder_post = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        # DECODER
        self.decoder_pre = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_pre_layers)])
        self.decoder_hourglass = DecoderHourglassBlock(cfg, cfg.shorten_factor, cfg.n_bottleneck_layers)
        self.decoder_post = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_post_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)
        
        # Output Head
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight # Weight tying

    def get_positional_embeddings(self, seq_len: int, device: torch.device):
        positions = torch.arange(seq_len, device=device).unsqueeze(0)
        return self.pos_emb(positions)

    def forward(self, enc_idx: torch.Tensor, dec_idx: torch.Tensor) -> torch.Tensor:
        B_enc, T_enc = enc_idx.shape
        B_dec, T_dec = dec_idx.shape
        
        # 1. ENCODER PASS
        enc_x = self.tok_emb(enc_idx) + self.get_positional_embeddings(T_enc, enc_idx.device)
        for block in self.encoder_pre:
            enc_x = block(enc_x)
        enc_x = self.encoder_hourglass(enc_x)
        for block in self.encoder_post:
            enc_x = block(enc_x)
        encoder_out = self.encoder_ln(enc_x)

        # 2. DECODER PASS
        dec_x = self.tok_emb(dec_idx) + self.get_positional_embeddings(T_dec, dec_idx.device)
        for block in self.decoder_pre:
            dec_x = block(dec_x, encoder_out)
        dec_x = self.decoder_hourglass(dec_x, encoder_out)
        for block in self.decoder_post:
            dec_x = block(dec_x, encoder_out)
            
        dec_x = self.decoder_ln(dec_x)
        logits = self.lm_head(dec_x)
        
        return logits

In [5]:
# --- Phase 1 c-3 notebook, cell 23 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    """Wrapper that keeps one forward(enc_idx, dec_idx) -> logits (B, T_dec, V) and pads the DECODER input on
    the right to a multiple of shorten_factor (safe: the decoder is causal, padding cannot change earlier logits).
    The ENCODER input is bidirectional, so it must already be a multiple of shorten_factor (never padded)."""
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = HourglassEncoderDecoder(cfg)          # YOUR model

    def forward(self, enc_idx, dec_idx):
        f = self.cfg.shorten_factor
        assert enc_idx.size(1) % f == 0, "encoder length must be a multiple of shorten_factor"
        T = dec_idx.size(1)
        pad = (-T) % f
        if pad:
            dec_idx = F.pad(dec_idx, (0, pad), value=0)    # id 0 = <pad>
        logits = self.core(enc_idx, dec_idx)
        return logits[:, :T, :]

In [6]:
# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell A: find and load the c-3 checkpoint and tokenizer

In [7]:
!find /kaggle/input -path "*phase-1-c-3*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;

-rw-rw-rw- 1 root root 476977 Oct  4 14:51 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 476977 Oct  4 14:51 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl
-rw-rw-rw- 1 root root 220615443 Oct  4 14:51 /kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full/model_encdec/model_final.pt


In [8]:
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, max_seq_len 1536, shorten 4
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-3-hierarchial-encoder-decoder/cs781_phase1_full"   # fix to what ls showed
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-3's own tokenizer (vocab 11022)
ck = torch.load(f"{ROOT}/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| max_seq_len", cfg.max_seq_len, "| shorten", cfg.shorten_factor)

<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | max_seq_len 1536 | shorten 4


## Cell B: per-token features from the encoder (sanity check should print max diff 0.0)

In [9]:
def features(idx):                        # idx: (B, T) LongTensor; T must be a multiple of 4 and <= 1536
    c = lm.core
    T = idx.size(1)
    assert T % c.cfg.shorten_factor == 0 and T <= c.cfg.max_seq_len
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_pre:
        x = blk(x)
    x = c.encoder_hourglass(x)            # 4 bottleneck layers on the 4x shortened sequence, upsampled back
    for blk in c.encoder_post:
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the encoder output inside the full model's forward
got = {}
h = lm.core.encoder_ln.register_forward_hook(lambda m, i, o: got.__setitem__("e", o))
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    lm(idx, torch.full((1, 4), 2, device=device)); f = features(idx)
h.remove(); print("max diff", (f - got["e"]).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)

max diff 0.0 (1, 64, 560)


## Cell C: Phase 5 LSI data

In [10]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 5 (LSI) data: load train / dev / test cases and the statute set
# One case = one long judgment -> a SET of statute IDs (multi-label).
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json
import numpy as np
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P5_DIR = "Phase_5_LSI"
print("Phase 5 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_5" in f])

def get5(name):
    """Reads Phase_5_LSI/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P5_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get5("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("labels", "label", "statutes", "statute_ids", "target", "y")

def _as_label_list(lab, k):
    """Any of [1, 7], "1,7", 7, None -> sorted list of unique ints."""
    if lab is None:
        return None
    if isinstance(lab, str):
        lab = [x for x in lab.replace(";", ",").split(",") if x.strip()]
    if not isinstance(lab, (list, tuple)):
        lab = [lab]
    out = sorted({int(x) for x in lab})
    return out

def as_cases(raw, name):
    """-> {case_id: {"text": str, "labels": [int, ...] or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("labels": [...])}.
    Repeated ids: exact duplicates dropped; differing repeats kept as "<id>#dup<n>" (train/dev),
    dropped with a warning on test (one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff = {}, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        rec = {"text": str(text), "labels": _as_label_list(d[lf] if lf else None, k)}
        if k in cases:
            if cases[k] == rec:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/labels kept {n_diff}")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get5("Train/dev.json"), "dev")
test  = as_cases(get5("Test/test.json"), "test")
del _raw

assert all(d["labels"] for s in (train, dev) for d in s.values()), \
    "every train/dev case needs at least one statute: check LABEL_FIELDS"
_tl = sum(d["labels"] is not None for d in test.values())
if _tl:
    print(f"test has a label field on {_tl} cases (ignored, set to None)")
for d in test.values():
    d["labels"] = None

# -------------------------------------------------------------------------------------
# Statute set = every ID seen in train or dev. Model output j <-> statute STATUTES[j].
# -------------------------------------------------------------------------------------
STATUTES = sorted({l for s in (train, dev) for d in s.values() for l in d["labels"]})
S2J = {s: j for j, s in enumerate(STATUTES)}
N_LABELS = len(STATUTES)
VALID_STATUTES = set(STATUTES)

tr_cnt = Counter(l for d in train.values() for l in d["labels"])
dv_cnt = Counter(l for d in dev.values() for l in d["labels"])
print(f"\n{N_LABELS} statutes | ID range {STATUTES[0]}..{STATUTES[-1]} | "
      f"in train {len(tr_cnt)} | in dev {len(dv_cnt)} | dev-only (never in train) {sorted(set(dv_cnt) - set(tr_cnt))}")
for name, split in (("train", train), ("dev", dev)):
    n = np.array([len(d["labels"]) for d in split.values()])
    print(f"{name:5s}: {len(split):6d} cases | statutes per case min {n.min()} mean {n.mean():.2f} "
          f"median {np.median(n):.0f} max {n.max()}")
print(f"test : {len(test):6d} cases")

f = np.array([tr_cnt.get(s, 0) for s in STATUTES])
for lo, hi in ((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9)):
    print(f"  statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {int(((f >= lo) & (f <= hi)).sum())}")
print("most frequent:", tr_cnt.most_common(10))
print("rarest       :", sorted(tr_cnt.items(), key=lambda x: x[1])[:10])
print("first test ids:", list(test)[:3])
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")

Phase 5 files on HF: ['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 42750 | first record keys: ['id', 'text', 'labels']
train: 42750 records -> 42750 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev: 10181 records -> 10181 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test: 13019 records -> 13019 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0

100 statutes | ID range 0..99 | in train 100 | in dev 100 | dev-only (never in train) []
train:  42750 cases | statutes per case min 1 mean 3.70 median 3 max 31
dev  :  10181 cases | statutes per case min 1 mean 3.92 median 3 max 28
test :  13019 cases
  statutes with 0-0 train cases: 0
  statutes with 1-5 train cases: 0
  statutes with 6-20 train cases: 0
  statutes with 21-100 train cases: 0
  statutes with 101-inf train cases: 100
most frequent: [(5, 11730), (34, 10380), (97, 7880), (43, 7728), (39, 7120), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
rarest       : [(84, 272), (25, 287), (23, 304), (92, 309), (66, 309), (59, 310), (50, 318), (74, 319), (27, 329), (15, 336)]
first test ids: ['100002997', '100004477', '1000086']
train∩dev ids 0 | train∩test 0


## Cell D: tokenise (cached), multi-label scorer, writer, top-K baseline

In [11]:
# =====================================================================================
# CELL D — [ANCILLARY] document tokenisation, multi-label scorer, prediction writer, baseline
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1

# Tokenising long judgments with the Phase 1 BPE is slow, so the ids are cached in /kaggle/working
# (re-tokenised automatically if Cell C gives different case ids).
import pickle
TOK_CACHE = "/kaggle/working/c3_lsi_ids.pkl"
_ids = pickle.load(open(TOK_CACHE, "rb")) if os.path.exists(TOK_CACHE) else None
if _ids is None or any(set(_ids.get(n, {})) != set(sp) for n, sp in (("train", train), ("dev", dev), ("test", test))):
    _ids = {}
    for name, split in (("train", train), ("dev", dev), ("test", test)):
        _ids[name] = {k: (tok.encode(d["text"]) or [UNK_ID]) for k, d in tqdm(split.items(), desc=f"tokenising {name}")}
    pickle.dump(_ids, open(TOK_CACHE, "wb"))
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for k, d in split.items():
        d["ids"] = _ids[name][k]
del _ids

# Length statistics in c-3 tokens, to choose WINDOW (multiple of 4, <= 1536), STRIDE and MAX_WINDOWS in Cell E1
for name, split in (("train", train), ("dev", dev), ("test", test)):
    L = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {L.min()} median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
          f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 1536: {(L > 1536).mean():.1%}")


# -------------------------------------------------------------------------------------
# Local scorer. pred / gold are SETS of statute IDs per case.
#   macro-F1 (primary): mean per-statute F1 over the statutes in the REFERENCE (gold) labels of the
#                       scored split, the way the spec says the evaluator sets the label dimension.
#                       A statute that is never gold but is predicted only costs micro precision.
#   macro-F1 (all):     the same mean over every statute in STATUTES (stricter; shown for reference).
#   micro-F1, subset accuracy (exact set match): secondary metrics.
# -------------------------------------------------------------------------------------
def ml_scores(gold_sets, pred_sets):
    tp, fp, fn = Counter(), Counter(), Counter()
    exact = 0
    for g, p in zip(gold_sets, pred_sets):
        g, p = set(g), set(p)
        exact += g == p
        for s in g & p: tp[s] += 1
        for s in p - g: fp[s] += 1
        for s in g - p: fn[s] += 1
    def f1(s):
        P = tp[s] / (tp[s] + fp[s]) if tp[s] + fp[s] else 0.0
        R = tp[s] / (tp[s] + fn[s]) if tp[s] + fn[s] else 0.0
        return P, R, (2 * P * R / (P + R) if P + R else 0.0), tp[s] + fn[s]
    ref = sorted({s for g in gold_sets for s in g})
    per = {s: f1(s) for s in sorted(set(ref) | set(STATUTES))}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mP = TP / (TP + FP) if TP + FP else 0.0
    mR = TP / (TP + FN) if TP + FN else 0.0
    return {"f1": float(np.mean([per[s][2] for s in ref])),
            "f1_all": float(np.mean([per[s][2] for s in STATUTES])),
            "p": float(np.mean([per[s][0] for s in ref])), "r": float(np.mean([per[s][1] for s in ref])),
            "micro_f1": 2 * mP * mR / (mP + mR) if mP + mR else 0.0, "micro_p": mP, "micro_r": mR,
            "subset_acc": exact / max(1, len(gold_sets)), "per": per}

def evaluate(pred, docs, show=True):
    """pred: {case_id: [statute IDs]}. Scores every case in docs."""
    ids = list(docs)
    res = ml_scores([docs[k]["labels"] for k in ids], [pred[k] for k in ids])
    if show:
        n_pred = np.mean([len(pred[k]) for k in ids])
        print(f"macro-F1 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) | macro-F1 over all statutes {res['f1_all']:.4f} | "
              f"micro-F1 {res['micro_f1']:.4f} (P {res['micro_p']:.4f} R {res['micro_r']:.4f}) | "
              f"subset acc {res['subset_acc']:.4f} | statutes predicted per case {n_pred:.2f}")
    return res

def per_statute_report(res, k=15):
    """Per-statute F1 with train frequency: the rarest and the worst statutes (for the report)."""
    rows = [(s, *res["per"][s], tr_cnt.get(s, 0)) for s in STATUTES]
    print(f"{'statute':>8s} {'P':>6s} {'R':>6s} {'F1':>6s} {'dev n':>6s} {'train n':>8s}")
    for title, key in (("rarest in train", lambda r: r[5]), ("lowest F1 (dev n > 0)", lambda r: (r[3], -r[4]))):
        print(f"-- {title}")
        for s, P, R, F, n, ntr in sorted([r for r in rows if r[4] > 0], key=key)[:k]:
            print(f"{s:8d} {P:6.3f} {R:6.3f} {F:6.3f} {n:6d} {ntr:8d}")
    for lo, hi in ((1, 5), (6, 20), (21, 100), (101, 10**9)):
        f = [r[3] for r in rows if r[4] > 0 and lo <= r[5] <= hi]
        if f:
            print(f"statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {len(f)} | mean dev F1 {np.mean(f):.4f}")

try:   # cross-check the macro / micro numbers against sklearn once
    from sklearn.metrics import f1_score
    from sklearn.preprocessing import MultiLabelBinarizer
    _g, _p = [[STATUTES[0]], [STATUTES[0], STATUTES[-1]], [STATUTES[-1]]], [[STATUTES[0]], [STATUTES[0]], []]
    _m = MultiLabelBinarizer(classes=sorted({s for g in _g for s in g})).fit(_g)
    _r = ml_scores(_g, _p)
    assert abs(_r["f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="macro", zero_division=0)) < 1e-9
    assert abs(_r["micro_f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="micro", zero_division=0)) < 1e-9
    print("scorer matches sklearn macro / micro F1")
except ImportError:
    pass


# -------------------------------------------------------------------------------------
# Writer: prediction.json = LIST of {"id", "pred": [int, ...]} in test order, zipped.
# Checks: one entry per test id, plain Python ints, no duplicates, every ID a known statute.
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_docs, out_dir="/kaggle/working"):
    os.makedirs(out_dir, exist_ok=True)
    assert set(pred) >= set(test_docs), "missing test ids"
    out, n_empty = [], 0
    for case_id in test_docs:
        p = sorted({int(s) for s in pred[case_id]})      # numpy ints -> plain int, deduplicated
        bad = [s for s in p if s not in VALID_STATUTES]
        assert not bad, f"{case_id}: unknown statute IDs {bad}"
        n_empty += not p
        out.append({"id": case_id, "pred": p})
    assert len(out) == len(test_docs) and len({o["id"] for o in out}) == len(out), "one entry per test case"
    if n_empty:
        print(f"WARNING {n_empty} test cases have an empty pred list (the spec does not say if [] is accepted)")
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    back = json.load(open(path))
    assert isinstance(back, list) and all(set(o) == {"id", "pred"} and isinstance(o["pred"], list)
                                          and all(type(s) is int for s in o["pred"])
                                          and len(set(o["pred"])) == len(o["pred"]) for o in back)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    n = [len(o["pred"]) for o in out]
    print(f"{len(out)} cases | statutes per case mean {np.mean(n):.2f} max {max(n)} | "
          f"distinct statutes predicted {len({s for o in out for s in o['pred']})} -> {zpath}")


# -------------------------------------------------------------------------------------
# Baseline: the K most frequent train statutes for every case (K = median statutes per train case).
# Checks the scorer and writer end to end before any modelling.
# -------------------------------------------------------------------------------------
_K = int(np.median([len(d["labels"]) for d in train.values()]))
_top = [s for s, _ in tr_cnt.most_common(_K)]
print(f"baseline: top-{_K} statutes {_top} for every case")
evaluate({k: _top for k in dev}, dev)
write_prediction({k: _top for k in test}, test, out_dir="/kaggle/working/baseline")   # format check

tokenising train:   0%|          | 0/42750 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/10181 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/13019 [00:00<?, ?it/s]

train tokens: min 2 median 717 p90 4626 p99 14020 max 158155 | > 1536: 33.0%
dev   tokens: min 4 median 911 p90 5031 p99 15324 max 129542 | > 1536: 38.1%
test  tokens: min 2 median 899 p90 4790 p99 14810 max 256349 | > 1536: 36.9%
scorer matches sklearn macro / micro F1
baseline: top-3 statutes [5, 34, 97] for every case
macro-F1 0.0118 (P 0.0074 R 0.0300) | macro-F1 over all statutes 0.0118 | micro-F1 0.2131 (P 0.2456 R 0.1882) | subset acc 0.0002 | statutes predicted per case 3.00
13019 cases | statutes per case mean 3.00 max 3 | distinct statutes predicted 3 -> /kaggle/working/baseline/submission.zip


## Cell E0: c-3 stages, pad-free rows, frozen-layer cache, training loop, thresholds

In [12]:
# =====================================================================================
# CELL E0 — [ANCILLARY] plumbing for c-3 Phase 5 (LSI) partial / layer-wise fine-tuning
# Run once after Cells 0, A, B, C, D. Then run Cell E1.
# Needs from earlier cells: lm, cfg, device, features, SPECIAL_TOKENS, train, dev, test (with
#   d["ids"]), STATUTES, S2J, N_LABELS, tr_cnt, evaluate, write_prediction
#
# Why this is faster than LORMA / KD / full FT: the c-3 encoder is a chain of stages
#     embeddings -> pre0 -> pre1 -> hourglass (x4 shorter, 4 blocks) -> post0 -> post1 -> ln
# Partial fine-tuning trains only the stages from TRAIN_FROM upwards (your choice in E1); the
# stages below stay frozen. Frozen stages always give the same output for the same window, so
# this cell runs them ONCE over every train / dev window and keeps the result (the "cache").
# Every epoch after that runs only the trainable top stages + your head, forward and backward.
# (LORMA / LoRA / QLoRA freeze the weights too, but the gradient still has to flow back through
# every layer to reach the adapters, so each epoch costs a full forward + backward.)
#
# Windows, loss, aggregation, thresholds, scorer and writer are the same as in the c-6 Phase 5
# notebook, so the runs compare directly. Lines marked [c-3] are what the hourglass needs: every
# encoder row is a multiple of 4 tokens and there is no padding mask.
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import os, math, time, random, shutil
import numpy as np
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model    # 560
SF = cfg.shorten_factor   # [c-3] 4
PAD_ID = SPECIAL_TOKENS.index("<pad>")
CUDA = torch.cuda.is_available()

# The decoder is never used for classification; dropping it frees GPU memory. tok_emb stays (shared with lm_head).
lm.core.decoder_pre = None          # [c-3] the hourglass decoder has 4 parts
lm.core.decoder_hourglass = None
lm.core.decoder_post = None
lm.core.decoder_ln = None
if CUDA:
    torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# [c-3] encoder as named stages. STAGES lists them bottom to top; E1's TRAIN_FROM picks the
# first one that trains (everything below it is frozen and cached).
# FAST_ATTENTION: same maths as EncoderSelfAttention.forward, but in PyTorch's fused
# scaled_dot_product_attention (checked against the plain version at the end of this cell).
# -------------------------------------------------------------------------------------
FAST_ATTENTION = True

def _self_attn_fast(attn, x):
    B, T, C = x.shape
    qkv = attn.qkv_proj(x).chunk(3, dim=-1)
    q, k, v = [t.view(B, T, attn.n_heads, attn.d_head).transpose(1, 2) for t in qkv]
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=attn.attn_dropout.p if attn.training else 0.0)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return attn.resid_dropout(attn.out_proj(out))

def _block(blk, x):
    if FAST_ATTENTION:                          # EncoderBlock.forward with the fused attention
        x = x + _self_attn_fast(blk.self_attn, blk.ln1(x))
        return x + blk.ffn(blk.ln2(x))
    return blk(x)

STAGES = ([f"pre{i}" for i in range(cfg.n_pre_layers)] + ["hourglass"]
          + [f"post{i}" for i in range(cfg.n_post_layers)] + ["ln"])

def stage_module(backbone, name):
    c = backbone.core
    if name.startswith("pre"):
        return c.encoder_pre[int(name[3:])]
    if name.startswith("post"):
        return c.encoder_post[int(name[4:])]
    return c.encoder_hourglass if name == "hourglass" else c.encoder_ln

def run_stage(backbone, name, x):
    m = stage_module(backbone, name)
    if name == "hourglass":                     # downsample x4 -> bottleneck blocks -> upsample + residual
        s = m.downsample(x)
        for blk in m.bottleneck:
            s = _block(blk, s)
        return m.upsample(s) + x
    return m(x) if name == "ln" else _block(m, x)

def encoder_below(backbone, idx, first):
    """Token ids (R, L) -> hidden state entering stage STAGES[first] (embeddings + all stages below it)."""
    c = backbone.core
    L = idx.size(1)
    assert L <= c.cfg.max_seq_len and L % SF == 0, "[c-3] row length must be a multiple of 4, <= 1536"
    x = c.tok_emb(idx) + c.get_positional_embeddings(L, idx.device)
    for name in STAGES[:first]:
        x = run_stage(backbone, name, x)
    return x

def encoder_above(backbone, x, first):
    """Hidden state entering STAGES[first] -> encoder output (R, L, 560)."""
    for name in STAGES[first:]:
        x = run_stage(backbone, name, x)
    return x

def encoder_forward(backbone, idx):
    return encoder_above(backbone, encoder_below(backbone, idx, 0), 0)


def stage_params():
    """Parameters per stage (for choosing TRAIN_FROM in E1)."""
    for name in STAGES:
        n = sum(p.numel() for p in stage_module(lm, name).parameters())
        print(f"  {name:10s} {n / 1e6:7.3f}M params")
    print(f"  embeddings {sum(p.numel() for p in (lm.core.tok_emb.weight, lm.core.pos_emb.weight)) / 1e6:7.3f}M (always frozen here)")


# -------------------------------------------------------------------------------------
# [c-3] Rows without padding (as in Phases 2-4): a window of n tokens with n % 4 == 0 is one
# row; otherwise two overlapping rows of L = 4*floor(n/4) (first L and last L tokens), whose
# outputs are stitched back to exactly n token vectors (each token read once). Only n < 4
# gets <pad> (never seen by the head: the stitch cuts it off).
# -------------------------------------------------------------------------------------
def rows_for(ids):
    n = len(ids)
    L = (n // SF) * SF
    if L == n:
        return [list(ids)]
    if L == 0:
        return [list(ids) + [PAD_ID] * (SF - n)]
    return [list(ids[:L]), list(ids[n - L:])]

def stitch(hs, n):
    """hs: per row slot (B, L, D) -> (B, n, D) token vectors of the windows."""
    if len(hs) == 1:
        return hs[0][:, :n]
    L = hs[0].size(1); o = n - L                # row 1 starts at token o; tokens L..n-1 are its last o vectors
    return torch.cat([hs[0], hs[1][:, L - o:]], dim=1)


# -------------------------------------------------------------------------------------
# Windows + which windows stand for the case: same as the c-6 Phase 5 E0
#   DOC_REPR "all" = every window; or keep at most max_windows: "first", "last",
#   "first_last" (half from each end) or "spread" (evenly over the whole case).
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def build_examples(docs, window, stride, doc_repr="all", max_windows=None):
    """One example per window: {"doc", "w", "n", "rows", "labels"} (labels = the case's statute set, None for test)."""
    assert doc_repr in DOC_REPRS and window <= cfg.max_seq_len
    assert window % SF == 0, "[c-3] WINDOW must be a multiple of 4 (full windows are then one row each)"
    assert doc_repr == "all" or max_windows, "set max_windows for this doc_repr"
    examples = []
    for doc_id, d in tqdm(docs.items(), desc="windowing", leave=False):
        ids = d["ids"]
        wins = select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows)
        for w, (s, e) in enumerate(wins):
            examples.append({"doc": doc_id, "w": w, "n": e - s, "rows": rows_for(ids[s:e]), "labels": d["labels"]})
    per = Counter(ex["doc"] for ex in examples)
    print(f"{len(per)} cases -> {len(examples)} windows | windows per case: mean {len(examples)/len(per):.2f} "
          f"max {max(per.values())} | encoder rows {sum(len(ex['rows']) for ex in examples)}")
    return examples

def make_batches(examples, batch_size, shuffle):
    """Batches of windows with IDENTICAL length n (so their rows line up too; no padding)."""
    by_len = defaultdict(list)
    for ex in examples:
        by_len[ex["n"]].append(ex)
    out = []
    for exs in by_len.values():
        if shuffle:
            exs = random.sample(exs, len(exs))
        out += [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out

def multi_hot(statutes):
    v = torch.zeros(N_LABELS)
    v[[S2J[s] for s in statutes]] = 1.0
    return v

def to_tensors(batch, cache=None):
    """cache given: the frozen stages' outputs come from it ("h"); otherwise token ids ("rows")."""
    b = {"n": batch[0]["n"]}
    if cache is not None and cache.where != "none":
        b["h"] = cache.get(batch)
    else:
        b["rows"] = [torch.tensor([ex["rows"][r] for ex in batch], device=device) for r in range(len(batch[0]["rows"]))]
    if batch[0]["labels"] is not None:
        b["labels"] = torch.stack([multi_hot(ex["labels"]) for ex in batch]).to(device)   # (B, N_LABELS) 0/1
    return b


# -------------------------------------------------------------------------------------
# Model: frozen stages (no graph, eval mode) -> trainable stages -> stitch -> YOUR head
#   head: (B, n, 560) token vectors of B windows -> (B, N_LABELS) logits (same contract as c-6)
# -------------------------------------------------------------------------------------
class PartialLSIModel(nn.Module):
    def __init__(self, backbone, head, first):
        super().__init__()
        self.backbone, self.head, self.first = backbone, head, first

    def frozen_modules(self):
        return [self.backbone.core.tok_emb, self.backbone.core.pos_emb] + \
               [stage_module(self.backbone, s) for s in STAGES[:self.first]]

    def train(self, mode=True):
        super().train(mode)
        for m in self.frozen_modules():         # frozen stages never use dropout: same output as the cache
            m.eval()
        return self

    def forward(self, b):
        if "h" in b:
            hs = b["h"]
        else:
            with torch.no_grad():               # frozen part: no activations kept for backward
                hs = [encoder_below(self.backbone, r, self.first) for r in b["rows"]]
        hs = [encoder_above(self.backbone, h, self.first) for h in hs]
        return self.head(stitch(hs, b["n"]))


def set_trainable(backbone, first):
    """Freeze everything, then unfreeze the stages STAGES[first:]."""
    for p in backbone.parameters():
        p.requires_grad = False
    for name in STAGES[first:]:
        for p in stage_module(backbone, name).parameters():
            p.requires_grad = True

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Cache of the frozen stages' output for every row of every window, in fp16.
#   where "gpu" (fastest) | "cpu" (RAM) | "disk" (np.memmap file in cache_dir) | "auto":
#   gpu if it fits in gpu_max_gb, else cpu if it fits in half the free RAM, else "none".
#   "none" = no cache: the frozen stages run again every step, but without backward. This is
#   faster than "disk", whose random reads of tens of GB are disk-bound (hours per epoch).
#   The size is printed before anything is computed: rows x tokens x 560 x 2 bytes.
# -------------------------------------------------------------------------------------
class FrozenCache:
    def __init__(self, model, examples, batch_size, where="auto", cache_dir="/kaggle/tmp", tag="train",
                 gpu_max_gb=5.0):
        n_tok = sum(len(r) for ex in examples for r in ex["rows"])
        gb = n_tok * FEAT_DIM * 2 / 2**30
        if where == "auto":
            try:
                import psutil; free_ram = psutil.virtual_memory().available / 2**30
            except ImportError:
                free_ram = 0.0
            where = "gpu" if CUDA and gb <= gpu_max_gb else "cpu" if gb <= free_ram / 2 else "none"
        print(f"cache [{tag}]: {len(examples)} windows, {n_tok / 1e6:.2f}M tokens -> {gb:.2f} GB fp16 in {where}")
        self.where, self.key = where, f"_off_{tag}"
        if where == "none":
            self.build_s = 0.0
            print(f"cache [{tag}]: off, the frozen stages run every step (forward only)")
            return
        if where == "disk":
            os.makedirs(cache_dir, exist_ok=True)
            free = shutil.disk_usage(cache_dir).free / 2**30
            assert gb < free - 1, f"cache needs {gb:.1f} GB but {cache_dir} has {free:.1f} GB free: fewer windows (MAX_WINDOWS) or another CACHE_DIR"
            self.path = os.path.join(cache_dir, f"c3_frozen_{tag}.f16")
            self.store = np.memmap(self.path, dtype=np.float16, mode="w+", shape=(max(1, n_tok), FEAT_DIM))
        else:
            self.store = torch.empty((max(1, n_tok), FEAT_DIM), dtype=torch.float16,
                                     device=device if where == "gpu" else "cpu",
                                     pin_memory=(where == "cpu" and CUDA))
        off = 0
        for ex in examples:
            ex[self.key] = []
            for r in ex["rows"]:
                ex[self.key].append(off); off += len(r)
        t = time.time()
        was = model.training
        model.eval()
        with torch.no_grad():
            for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc=f"caching {tag}", leave=False):
                b = to_tensors(batch)
                with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                    hs = [encoder_below(model.backbone, r, model.first) for r in b["rows"]]
                for ri, h in enumerate(hs):
                    h = h.to(torch.float16)
                    L = h.size(1)
                    for ex, v in zip(batch, h):
                        o = ex[self.key][ri]
                        if where == "disk":
                            self.store[o:o + L] = v.cpu().numpy()
                        else:
                            self.store[o:o + L] = v
        if where == "disk":
            self.store.flush()
        model.train(was)
        self.build_s = time.time() - t
        print(f"cache [{tag}] built in {self.build_s / 60:.1f} min")

    def get(self, batch):
        out = []
        for ri in range(len(batch[0]["rows"])):
            L = len(batch[0]["rows"][ri])
            offs = [ex[self.key][ri] for ex in batch]
            if self.where == "disk":
                h = torch.from_numpy(np.stack([self.store[o:o + L] for o in offs]))
            else:
                h = torch.stack([self.store[o:o + L] for o in offs])
            out.append(h.to(device, non_blocking=True).float())   # fp32 like the residual stream of the full path
        return out


# -------------------------------------------------------------------------------------
# Loss: same as c-6 (binary relevance, BCE against the multi-hot target, optional pos_weight)
#   make_pos_weight: "none" | "neg_pos" (#without / #with) | "sqrt_neg_pos"; cap = upper limit
# -------------------------------------------------------------------------------------
def bce_loss(logits, targets, pos_weight=None):
    return F.binary_cross_entropy_with_logits(logits.float(), targets, pos_weight=pos_weight)

def make_pos_weight(scheme, cap=None):
    assert scheme in ("none", "neg_pos", "sqrt_neg_pos"), "set POS_WEIGHT_SCHEME"
    if scheme == "none":
        return None
    n = len(train)
    pos = torch.tensor([max(1, tr_cnt.get(s, 0)) for s in STATUTES], dtype=torch.float)
    w = (n - pos) / pos
    if scheme == "sqrt_neg_pos":
        w = w.sqrt()
    if cap is not None:
        w = w.clamp(max=cap)
    print(f"pos_weight: min {w.min():.2f} median {w.median():.2f} max {w.max():.2f}")
    return w.to(device)


# -------------------------------------------------------------------------------------
# Prediction: per-window sigmoid probabilities -> one vector per case ("mean" / "max") -> statute set
#   decide: threshold = one float (global) or an array of N_LABELS (per statute); min_labels = top-k fallback
# -------------------------------------------------------------------------------------
AGGS = ("mean", "max")

@torch.no_grad()
def predict_probs(model, examples, batch_size, agg, cache=None):
    """-> {case_id: np.array (N_LABELS,)}. cache: that split's FrozenCache (faster), or None (full path)."""
    assert agg in AGGS, agg
    model.eval()
    per_doc = defaultdict(list)
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            pr = model(to_tensors(batch, cache)).float().sigmoid().cpu().numpy()
        for ex, p in zip(batch, pr):
            per_doc[ex["doc"]].append(p)
    f = np.mean if agg == "mean" else np.max
    return {d: f(v, axis=0) for d, v in per_doc.items()}

def decide(probs, threshold, min_labels=0):
    thr = np.broadcast_to(np.asarray(threshold, dtype=float), (N_LABELS,))
    out = {}
    for d, p in probs.items():
        js = np.flatnonzero(p >= thr)
        if len(js) < min_labels:
            js = np.argsort(-p)[:min_labels]
        out[d] = [STATUTES[j] for j in js]
    return out

def predict(model, examples, batch_size, agg, threshold, min_labels=0, cache=None):
    return decide(predict_probs(model, examples, batch_size, agg, cache), threshold, min_labels)

def tune_thresholds(probs, docs, mode, grid, min_labels=0):
    """Same as c-6: "global" = best single grid value on dev; "per_label" = then per statute."""
    assert mode in ("global", "per_label") and grid is not None and len(grid), "set THRESH_MODE / THRESH_GRID"
    ids = list(docs)
    P = np.stack([probs[k] for k in ids])
    Y = np.stack([multi_hot(docs[k]["labels"]).numpy() for k in ids]).astype(bool)
    scores = [(evaluate(decide(probs, t, min_labels), docs, show=False)["f1"], t) for t in grid]
    best_f1, best_t = max(scores)
    print("global:", " ".join(f"{t:.2f}->{f:.4f}" for f, t in scores), f"| best {best_t} (dev macro-F1 {best_f1:.4f})")
    if mode == "global":
        return float(best_t)
    thr = np.full(N_LABELS, float(best_t))
    for j in range(N_LABELS):
        if not Y[:, j].any():
            continue
        best = (-1.0, best_t)
        for t in grid:
            pj = P[:, j] >= t
            tp = (pj & Y[:, j]).sum(); fp = (pj & ~Y[:, j]).sum(); fn = (~pj & Y[:, j]).sum()
            best = max(best, (2 * tp / (2 * tp + fp + fn) if tp else 0.0, t))
        thr[j] = best[1]
    f = evaluate(decide(probs, thr, min_labels), docs, show=False)["f1"]
    print(f"per_label: dev macro-F1 {f:.4f} | thresholds min {thr.min():.2f} median {np.median(thr):.2f} max {thr.max():.2f}")
    return thr


# -------------------------------------------------------------------------------------
# Training loop (c-6 train_model, reading the cache). Keeps the epoch with the best dev macro-F1
# at the fixed threshold. Logs the wall-clock minutes of every epoch (training part and dev part).
# -------------------------------------------------------------------------------------
def train_model(model, train_ex, val_ex, val_docs, loss_fn, *, lr, epochs, batch_size, agg, threshold,
                min_labels=0, weight_decay=0.0, grad_clip=1.0, lr_lambda=None, head_lr=None,
                save_path="best_trainable.pt", max_epoch_minutes=None, train_cache=None, val_cache=None):
    """head_lr: optional separate LR for the head (None = lr). train_cache / val_cache: FrozenCache
    objects (None = run the frozen stages every time, still without backward through them)."""
    groups = [{"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": lr},
              {"params": [p for p in model.head.parameters() if p.requires_grad], "lr": head_lr or lr}]
    groups = [g for g in groups if g["params"]]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log, skipped = time.time(), -1.0, None, [], 0
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {epoch + 1}/{epochs}")
        t_ep, seen = time.time(), 0
        for batch in pbar:
            if max_epoch_minutes and time.time() - t_ep > 60 * max_epoch_minutes:
                tqdm.write(f"epoch {epoch + 1}: time cap {max_epoch_minutes} min reached after {seen}/{len(batches)} batches")
                break
            seen += 1
            b = to_tensors(batch, train_cache)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                logits = model(b)
            loss = loss_fn(logits.float(), b)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        if CUDA:
            torch.cuda.synchronize()
        train_min = (time.time() - t_ep) / 60
        t_dev = time.time()
        res = evaluate(predict(model, val_ex, batch_size, agg, threshold, min_labels, val_cache), val_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "batches": seen, "of": len(batches),
                    "train_min": round(train_min, 2), "dev_min": round((time.time() - t_dev) / 60, 2),
                    "dev_macro_f1": res["f1"], "dev_micro_f1": res["micro_f1"],
                    "dev_subset_acc": res["subset_acc"], "skipped": skipped})
        print(log[-1])
        if res["f1"] > best_f1:
            best_f1, best_res = res["f1"], {k: v for k, v in res.items() if k != "per"}
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    peak = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float("nan")
    print(f"best dev macro-F1 {best_f1:.4f} (micro-F1 {best_res['micro_f1']:.4f}, subset acc {best_res['subset_acc']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | mean training min/epoch "
          f"{np.mean([l['train_min'] for l in log]):.2f} | peak GPU mem {peak:.2f} GB")
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, batch_size, agg, cache=None):
    if CUDA: torch.cuda.synchronize()
    t = time.time()
    predict_probs(model, examples, batch_size, agg, cache)
    if CUDA: torch.cuda.synchronize()
    return time.time() - t


def step_time_vs_full_ft(model, examples, batch_size, cache, loss_fn, reps=5):
    """Seconds per training step (forward + backward, no optimizer) on the same batches:
    (a) this partial setup reading the cache, (b) the same model with EVERY encoder stage
    trainable and no cache (= full fine-tuning cost). Weights are not changed; grads are cleared.
    The ratio is the speed-up to report; (b) x steps per epoch estimates a full-FT epoch."""
    batches = [b for b in make_batches(examples, batch_size, shuffle=False) if b[0]["n"] == max(ex["n"] for ex in examples)][:reps]
    def run(use_cache):
        model.train()
        if CUDA: torch.cuda.synchronize()
        t = time.time()
        for batch in batches:
            b = to_tensors(batch, cache if use_cache else None)
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                loss_fn(model(b).float(), b).backward()
        if CUDA: torch.cuda.synchronize()
        return (time.time() - t) / len(batches)
    run(True)                                   # warm-up
    t_part = run(True)
    keep = {n: p.requires_grad for n, p in model.named_parameters()}
    first = model.first
    model.first = 0; set_trainable(model.backbone, 0)   # all 8 blocks + ln train (embeddings stay frozen)
    try:
        run(False); t_full = run(False)
    except torch.cuda.OutOfMemoryError:
        t_full = float("nan"); print("full-FT timing ran out of GPU memory at this batch size")
    model.first = first
    for n, p in model.named_parameters():
        p.requires_grad = keep[n]; p.grad = None
    if CUDA: torch.cuda.empty_cache()
    print(f"step time on {len(batches)} batches of {batch_size} x {batches[0][0]['n']} tokens: partial (cached) "
          f"{t_part:.3f}s | full fine-tuning {t_full:.3f}s -> {t_full / t_part:.1f}x faster per step")
    return t_part, t_full


def error_cases(pred, docs, k=5, chars=300):
    """Statutes most often missed / wrongly added on dev, and a few cases with both kinds of error."""
    missed, added = Counter(), Counter()
    for i, d in docs.items():
        g, p = set(d["labels"]), set(pred[i])
        missed.update(g - p); added.update(p - g)
    print("most often MISSED (statute: times):", missed.most_common(15))
    print("most often WRONGLY ADDED (statute: times):", added.most_common(15))
    bad = sorted(docs, key=lambda i: -len(set(docs[i]["labels"]) ^ set(pred[i])))[:k]
    for i in bad:
        g, p = set(docs[i]["labels"]), set(pred[i])
        print(f"- {i} | {len(docs[i]['ids'])} tokens | gold {sorted(g)} | pred {sorted(p)} | "
              f"missed {sorted(g - p)} added {sorted(p - g)}\n  start: {docs[i]['text'][:chars]!r}")


def write_submission(model, build_kwargs, batch_size, agg, threshold, min_labels=0, best_path=None,
                     out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict test (full path, no cache), write prediction.json + zip."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, batch_size, agg, threshold, min_labels), test, out_dir=out_dir)


# -------------------------------------------------------------------------------------
# Checks: stage-wise encoder == Cell B features(); fused attention == plain attention
# -------------------------------------------------------------------------------------
lm.eval()
with torch.no_grad():
    _ids = next(iter(train.values()))["ids"][:64]
    _p = torch.tensor([_ids + [PAD_ID] * (64 - len(_ids))], device=device)
    FAST_ATTENTION = False; _s = encoder_forward(lm, _p)
    _ref = (_s - features(_p)).abs().max().item()
    FAST_ATTENTION = True; _f = encoder_forward(lm, _p)
_diff = (_f - _s).abs().max().item()
FAST_ATTENTION = _diff < 1e-3
print(f"stages vs Cell B features max diff {_ref:.2e} (expect 0) | fused attention max diff {_diff:.2e} "
      f"-> FAST_ATTENTION = {FAST_ATTENTION}")
assert _ref < 1e-4, "stage-wise encoder differs from Cell B"
print("stages:", STAGES); stage_params()


stages vs Cell B features max diff 0.00e+00 (expect 0) | fused attention max diff 9.54e-07 -> FAST_ATTENTION = True
stages: ['pre0', 'pre1', 'hourglass', 'post0', 'post1', 'ln']
  pre0         2.378M params
  pre1         2.378M params
  hourglass    9.513M params
  post0        2.378M params
  post1        2.378M params
  ln           0.001M params
  embeddings   7.032M (always frozen here)


## Cell E2: LoRA template (write LoRALinear, fill the LoRA values)

In [13]:
# =====================================================================================
# CELL E2 — LoRA on the c-3 hourglass encoder for LSI  <<< YOUR CELL (template) >>>
# Run after Cells 0, A, B, C, D and E0 (not after E1). To run again, re-run Cell A and E0 first
# (wrapping replaces the encoder's linears in place, and wrap_linears refuses to wrap twice).
#
# Note for the report: plain LoRA is NOT on the Phase 5 technique list (section 11.2.1); QLoRA is.
# This LoRA run is still a valid experiment/submission, but it does not count as one of the
# required distinct techniques unless the TAs say otherwise.
#
# Speed: LoRA adapters go only into the stages from LORA_FROM upwards. The stages below have no
# adapters and stay exactly as in Phase 1, so E0 runs them ONCE and caches them (same trick as E1).
# LORA_FROM = "pre0" puts adapters everywhere (no saving); higher = faster epochs, fewer adapters.
#
# YOURS:  [H] the LoRA values (None)   [L] LoRALinear (raises NotImplementedError until you write it)
# Pre-set (change as you like): data / loss / threshold values = your c-6 Phase 5 values (23 Oct),
# so the runs compare directly; [N] head = your c-6 Phase 5 mean+max LSIHead, unchanged.
# Sections marked [ANCILLARY] are plumbing; nothing to change there.
# =====================================================================================
import os, json

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — yours
# -------------------------------------------------------------------------------------
LORA_FROM     = "hourglass"  # first stage that gets adapters: "pre0" "pre1" "hourglass" "post0" "post1"
LORA_RANK     = 8
LORA_ALPHA    = 16  # if your layer scales BA by alpha / rank; None if it has no scaling
LORA_TARGETS  = ("qkv_proj", "out_proj", "ffn_in", "ffn_out")  # tuple, any of "qkv_proj", "out_proj", "ffn_in", "ffn_out" (sizes printed below)
LORA_DROPOUT  = 0.05  # if your layer uses dropout on its input; None if not
LR            = 3e-4
HEAD_LR       = None  # optional separate LR for the head (None = LR)
EPOCHS        = 9
MAX_EPOCH_MINUTES = None
WINDOW        = 512   # your c-6 value; [c-3] must be a multiple of 4, <= 1536
STRIDE        = 128   # your c-6 value
DOC_REPR      = "spread"   # your c-6 value
MAX_WINDOWS   = 8     # your c-6 value
BATCH_SIZE    = 16    # your c-6 value
WEIGHT_DECAY  = 0.01  # your c-6 value
LR_LAMBDA     = None  # optional: step -> LR multiplier, None = constant LR
POS_WEIGHT_SCHEME = "sqrt_neg_pos"   # your c-6 value
POS_WEIGHT_CAP    = 10.0             # your c-6 value
AGG           = "max" # your c-6 value
THRESHOLD     = 0.5   # your c-6 value
MIN_LABELS    = 0     # your c-6 value
THRESH_MODE   = "global"   # your c-6 value
THRESH_GRID   = [0.10, 0.15, 0.20, 0.25, 0.30,
                 0.35, 0.40, 0.45, 0.50, 0.55,
                 0.60, 0.65, 0.70, 0.75, 0.80]
SEED          = 42
# [ANCILLARY] frozen-stage cache (see E0): "auto" | "gpu" | "cpu" | "disk"
CACHE_WHERE      = "auto"
CACHE_GPU_MAX_GB = 5.0
CACHE_DIR        = "/kaggle/tmp"
TIME_VS_FULL_FT  = True
for _n in ["LORA_FROM", "LORA_RANK", "LORA_TARGETS", "LR", "EPOCHS", "WINDOW", "STRIDE", "DOC_REPR",
           "BATCH_SIZE", "WEIGHT_DECAY", "POS_WEIGHT_SCHEME", "AGG", "THRESHOLD", "MIN_LABELS", "THRESH_MODE"]:
    assert globals()[_n] is not None, f"set {_n}"
assert LORA_FROM in STAGES[:-1], f"LORA_FROM must be one of {STAGES[:-1]} (the final ln has no linears)"
assert set(LORA_TARGETS) <= {"qkv_proj", "out_proj", "ffn_in", "ffn_out"}, LORA_TARGETS
assert STRIDE <= WINDOW <= cfg.max_seq_len and WINDOW % SF == 0, "[c-3] WINDOW: multiple of 4, <= 1536"
assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup + which linears exist in the adapted stages
# -------------------------------------------------------------------------------------
FIRST = STAGES.index(LORA_FROM)
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c3_lora_{LORA_FROM}_r{LORA_RANK}_lsi_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
POS_WEIGHT = make_pos_weight(POS_WEIGHT_SCHEME, POS_WEIGHT_CAP)

def stage_blocks(name):
    """EncoderBlocks inside one stage (the hourglass holds the 4 bottleneck blocks)."""
    m = stage_module(lm, name)
    return list(m.bottleneck) if name == "hourglass" else [] if name == "ln" else [m]

def encoder_linears(stages, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every block of the given stages.
    c-3 attention has ONE fused qkv_proj (560 -> 1680) and no biases; the FFN linears have biases."""
    for name in stages:
        for blk in stage_blocks(name):
            for w in which:
                if w in ("qkv_proj", "out_proj"):
                    parent, attr = blk.self_attn, w
                else:
                    parent, attr = blk.ffn.net, "0" if w == "ffn_in" else "2"
                yield parent, attr, getattr(parent, attr)

def wrap_linears(stages, which, make_wrapper):
    todo = list(encoder_linears(stages, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

for _name in ("qkv_proj", "out_proj", "ffn_in", "ffn_out"):
    _, _, _lin = next(encoder_linears(["pre0"], (_name,)))
    print(f"  {_name:10s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")
print(f"adapted stages: {STAGES[FIRST:-1]} ({sum(len(stage_blocks(s)) for s in STAGES[FIRST:])} blocks) | "
      f"frozen + cached: embeddings, {STAGES[:FIRST]}")


# -------------------------------------------------------------------------------------
# [L] LoRA layer — yours
# -------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """Wraps ONE pretrained nn.Linear of the c-3 encoder.
    Constructed as   LoRALinear(base, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)   by wrap_linears below:
        base    : the original nn.Linear (weight (out_features, in_features), bias may be None)
        rank, alpha, dropout : your [H] values (alpha / dropout may be None if you don't use them)
    forward(x)
        x      : (B, T, in_features)
        return : (B, T, out_features)   SAME shape as base(x): it goes straight back into the block
    What the plumbing expects:
        - the pretrained weights stay frozen (requires_grad False); only your new parameters train.
          Everything with requires_grad True is optimised and saved to best_trainable.pt.
        - keep self.base as an attribute.
        - if your initialisation makes the layer start out equal to base, the identity check
          below passes (diff ~0). If your design does not, set CHECK_IDENTITY = False.
    """
    def __init__(self, base: nn.Linear, rank: int, alpha=None, dropout=None):
        super().__init__()
        self.base = base
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = (alpha / rank) if alpha is not None else 1.0
        self.lora_dropout = nn.Dropout(dropout) if dropout is not None and dropout > 0 else nn.Identity()

        for p in self.base.parameters():
            p.requires_grad = False

        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features))
        self.lora_B = nn.Parameter(torch.empty(self.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        adapter_in = self.lora_dropout(x)
        adapter_out = F.linear(F.linear(adapter_in, self.lora_A), self.lora_B)
        return base_out + self.scaling * adapter_out

# [ANCILLARY] quick shape test on a dummy linear (no model involved)
_t = LoRALinear(nn.Linear(FEAT_DIM, 3 * FEAT_DIM, bias=False), LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
_y = _t(torch.randn(2, 8, FEAT_DIM))
assert _y.shape == (2, 8, 3 * FEAT_DIM), f"LoRALinear output shape {tuple(_y.shape)}, expected (2, 8, {3 * FEAT_DIM})"
assert not any(p.requires_grad for p in _t.base.parameters()), "freeze the base linear inside LoRALinear"
print(f"LoRALinear trainable params on a {FEAT_DIM}->{3 * FEAT_DIM} layer:", sum(p.numel() for p in _t.parameters() if p.requires_grad))
del _t, _y


# -------------------------------------------------------------------------------------
# [N] Head — yours (c-6 Phase 5 mean + max pool head, unchanged)
# -------------------------------------------------------------------------------------
class LSIHead(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__()
        self.norm = nn.LayerNorm(2 * in_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim, in_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(in_dim, n_labels)
        )

    def forward(self, h):
        mean_pool = h.mean(dim=1)
        max_pool = h.max(dim=1).values
        pooled = torch.cat([mean_pool, max_pool], dim=-1)
        return self.mlp(self.norm(pooled))


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-3, wrap the chosen linears, identity check, model
# -------------------------------------------------------------------------------------
CHECK_IDENTITY = True
lm.eval()
_ids = next(iter(train.values()))["ids"][:64]
probe = torch.tensor([_ids + [PAD_ID] * (64 - len(_ids))], device=device)
with torch.no_grad():
    ref = encoder_forward(lm, probe).float()
for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(STAGES[FIRST:], LORA_TARGETS,
                         lambda lin: LoRALinear(lin, LORA_RANK, LORA_ALPHA, LORA_DROPOUT).to(device))
print(f"wrapped {n_wrapped} linears")
if CHECK_IDENTITY:
    with torch.no_grad():
        diff = (encoder_forward(lm, probe).float() - ref).abs().max().item()
    print(f"LoRA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
    assert diff < 1e-3, "wrapped model differs from the original at init"
assert not any(p.requires_grad for s in STAGES[:FIRST] for p in stage_module(lm, s).parameters()), \
    "a stage below LORA_FROM has trainable weights: the cache would be wrong"

head = LSIHead(FEAT_DIM, N_LABELS)
model = PartialLSIModel(lm, head, FIRST).to(device)
n_train_params, n_total_params = count_params(model)


# -------------------------------------------------------------------------------------
# [ANCILLARY] data, cache, checks, training, threshold tuning, scoring, submission
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(dev, **BUILD)
train_cache = FrozenCache(model, train_ex, BATCH_SIZE, CACHE_WHERE, CACHE_DIR, "train", CACHE_GPU_MAX_GB)
val_cache = FrozenCache(model, val_ex, BATCH_SIZE, CACHE_WHERE, CACHE_DIR, "dev", CACHE_GPU_MAX_GB)

model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
    for _b in make_batches(val_ex, 4, shuffle=False)[:3]:
        _d = (model(to_tensors(_b, val_cache)).float() - model(to_tensors(_b)).float()).abs().max().item()
        print(f"cached vs full path, {len(_b)} windows x {_b[0]['n']} tokens: max |logit diff| {_d:.2e}")
        assert _d < 5e-2, "cache does not match the frozen stages"

loss_fn = lambda logits, b: bce_loss(logits, b["labels"], POS_WEIGHT)
if TIME_VS_FULL_FT:
    t_step_part, t_step_full = step_time_vs_full_ft(model, train_ex, BATCH_SIZE, train_cache, loss_fn)

log, best_res = train_model(
    model, train_ex, val_ex, dev, loss_fn,
    lr=LR, head_lr=HEAD_LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD, min_labels=MIN_LABELS,
    max_epoch_minutes=MAX_EPOCH_MINUTES, train_cache=train_cache, val_cache=val_cache)
train_mem = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float("nan")

model.load_state_dict(torch.load(SAVE), strict=False)
dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG, val_cache)
print(f"\n-- dev at the fixed THRESHOLD {THRESHOLD}")
evaluate(decide(dev_probs, THRESHOLD, MIN_LABELS), dev)
FINAL_THR = THRESHOLD if THRESH_MODE == "fixed" else tune_thresholds(dev_probs, dev, THRESH_MODE, THRESH_GRID, MIN_LABELS)
dev_pred = decide(dev_probs, FINAL_THR, MIN_LABELS)
print(f"\n-- dev with the final thresholds ({THRESH_MODE})")
res = evaluate(dev_pred, dev)
per_statute_report(res)
error_cases(dev_pred, dev)

summary = {"run": RUN_NAME, "technique": f"LoRA from {LORA_FROM}", "rank": LORA_RANK, "alpha": LORA_ALPHA,
           "targets": list(LORA_TARGETS), "n_wrapped": n_wrapped, "build": BUILD,
           "trainable_M": round(n_train_params / 1e6, 3), "total_M": round(n_total_params / 1e6, 2),
           "cache_build_min": round((train_cache.build_s + val_cache.build_s) / 60, 2),
           "cache_where": train_cache.where,
           "train_min_per_epoch": round(float(np.mean([l["train_min"] for l in log])), 2),
           "dev_min_per_epoch": round(float(np.mean([l["dev_min"] for l in log])), 2),
           "step_s_lora": round(t_step_part, 3) if TIME_VS_FULL_FT else None,
           "step_s_full_ft": round(t_step_full, 3) if TIME_VS_FULL_FT else None,
           "dev_infer_s_full_path": round(time_inference(model, val_ex, BATCH_SIZE, AGG), 1),
           "peak_train_gb": round(train_mem, 2), "thresh_mode": THRESH_MODE,
           "dev": {k: round(res[k], 4) for k in ("f1", "f1_all", "micro_f1", "subset_acc")}}
print(summary)
json.dump({**summary, "log": log, "final_thr": np.asarray(FINAL_THR).tolist(),
           "dev_probs": {k: v.round(4).tolist() for k, v in dev_probs.items()}},
          open(f"{OUT_DIR}/dev_log.json", "w"))

write_submission(model, BUILD, BATCH_SIZE, AGG, FINAL_THR, MIN_LABELS, best_path=SAVE, out_dir=OUT_DIR)
for _c in (train_cache, val_cache):          # free the cache (GPU / RAM / file in CACHE_DIR)
    if _c.where == "disk":
        del _c.store; os.remove(_c.path)
del train_cache, val_cache, _c
if CUDA: torch.cuda.empty_cache()

pos_weight: min 1.63 median 7.84 max 10.00
  qkv_proj     560 ->  1680  bias=False
  out_proj     560 ->   560  bias=False
  ffn_in       560 ->  1000  bias=True
  ffn_out     1000 ->   560  bias=True
adapted stages: ['hourglass', 'post0', 'post1'] (6 blocks) | frozen + cached: embeddings, ['pre0', 'pre1']
LoRALinear trainable params on a 560->1680 layer: 17920
wrapped 24 linears
LoRA identity-at-init max |diff| = 0.00e+00 (should be ~0)
trainable 0.997M / total 27.06M (3.69%)


windowing:   0%|          | 0/42750 [00:00<?, ?it/s]

42750 cases -> 181027 windows | windows per case: mean 4.23 max 8 | encoder rows 193701


windowing:   0%|          | 0/10181 [00:00<?, ?it/s]

10181 cases -> 46998 windows | windows per case: mean 4.62 max 8 | encoder rows 49714
cache [train]: 181027 windows, 91.86M tokens -> 95.82 GB fp16 in none
cache [train]: off, the frozen stages run every step (forward only)
cache [dev]: 46998 windows, 23.82M tokens -> 24.84 GB fp16 in none
cache [dev]: off, the frozen stages run every step (forward only)
cached vs full path, 4 windows x 422 tokens: max |logit diff| 0.00e+00
cached vs full path, 1 windows x 422 tokens: max |logit diff| 0.00e+00
cached vs full path, 4 windows x 122 tokens: max |logit diff| 0.00e+00
step time on 5 batches of 16 x 512 tokens: partial (cached) 0.115s | full fine-tuning 0.161s -> 1.4x faster per step


epoch 1/9:   0%|          | 0/11549 [00:00<?, ?it/s]

... (30 outputs left out)

windowing:   0%|          | 0/13019 [00:00<?, ?it/s]

13019 cases -> 60010 windows | windows per case: mean 4.61 max 8 | encoder rows 63419


predict:   0%|          | 0/3988 [00:00<?, ?it/s]

WARNING 1118 test cases have an empty pred list (the spec does not say if [] is accepted)
13019 cases | statutes per case mean 7.52 max 37 | distinct statutes predicted 100 -> /kaggle/working/c3_lora_hourglass_r8_lsi_spread/submission.zip


In [14]:
# # =====================================================================================
# # CELL E2 — LoRA on the c-3 hourglass encoder for LSI  <<< YOUR CELL (template) >>>
# # Run after Cells 0, A, B, C, D and E0 (not after E1). To run again, re-run Cell A and E0 first
# # (wrapping replaces the encoder's linears in place, and wrap_linears refuses to wrap twice).
# #
# # Note for the report: plain LoRA is NOT on the Phase 5 technique list (section 11.2.1); QLoRA is.
# # This LoRA run is still a valid experiment/submission, but it does not count as one of the
# # required distinct techniques unless the TAs say otherwise.
# #
# # Speed: LoRA adapters go only into the stages from LORA_FROM upwards. The stages below have no
# # adapters and stay exactly as in Phase 1, so E0 runs them ONCE and caches them (same trick as E1).
# # LORA_FROM = "pre0" puts adapters everywhere (no saving); higher = faster epochs, fewer adapters.
# #
# # YOURS:  [H] the LoRA values (None)   [L] LoRALinear (raises NotImplementedError until you write it)
# # Pre-set (change as you like): data / loss / threshold values = your c-6 Phase 5 values (23 Oct),
# # so the runs compare directly; [N] head = your c-6 Phase 5 mean+max LSIHead, unchanged.
# # Sections marked [ANCILLARY] are plumbing; nothing to change there.
# # =====================================================================================
# import os, json

# # -------------------------------------------------------------------------------------
# # [H] Hyperparameters and design choices — yours
# # -------------------------------------------------------------------------------------
# LORA_FROM     = None  # first stage that gets adapters: "pre0" "pre1" "hourglass" "post0" "post1"
# LORA_RANK     = None
# LORA_ALPHA    = None  # if your layer scales BA by alpha / rank; None if it has no scaling
# LORA_TARGETS  = None  # tuple, any of "qkv_proj", "out_proj", "ffn_in", "ffn_out" (sizes printed below)
# LORA_DROPOUT  = None  # if your layer uses dropout on its input; None if not
# LR            = None
# HEAD_LR       = None  # optional separate LR for the head (None = LR)
# EPOCHS        = None
# MAX_EPOCH_MINUTES = None
# WINDOW        = 512   # your c-6 value; [c-3] must be a multiple of 4, <= 1536
# STRIDE        = 256   # your c-6 value
# DOC_REPR      = "spread"   # your c-6 value
# MAX_WINDOWS   = 8     # your c-6 value
# BATCH_SIZE    = 16    # your c-6 value
# WEIGHT_DECAY  = 0.01  # your c-6 value
# LR_LAMBDA     = None  # optional: step -> LR multiplier, None = constant LR
# POS_WEIGHT_SCHEME = "sqrt_neg_pos"   # your c-6 value
# POS_WEIGHT_CAP    = 10.0             # your c-6 value
# AGG           = "max" # your c-6 value
# THRESHOLD     = 0.5   # your c-6 value
# MIN_LABELS    = 0     # your c-6 value
# THRESH_MODE   = "global"   # your c-6 value
# THRESH_GRID   = [0.10, 0.15, 0.20, 0.25, 0.30,
#                  0.35, 0.40, 0.45, 0.50, 0.55,
#                  0.60, 0.65, 0.70, 0.75, 0.80]
# SEED          = 42
# # [ANCILLARY] frozen-stage cache (see E0): "auto" | "gpu" | "cpu" | "disk"
# CACHE_WHERE      = "auto"
# CACHE_GPU_MAX_GB = 5.0
# CACHE_DIR        = "/kaggle/tmp"
# TIME_VS_FULL_FT  = True
# for _n in ["LORA_FROM", "LORA_RANK", "LORA_TARGETS", "LR", "EPOCHS", "WINDOW", "STRIDE", "DOC_REPR",
#            "BATCH_SIZE", "WEIGHT_DECAY", "POS_WEIGHT_SCHEME", "AGG", "THRESHOLD", "MIN_LABELS", "THRESH_MODE"]:
#     assert globals()[_n] is not None, f"set {_n}"
# assert LORA_FROM in STAGES[:-1], f"LORA_FROM must be one of {STAGES[:-1]} (the final ln has no linears)"
# assert set(LORA_TARGETS) <= {"qkv_proj", "out_proj", "ffn_in", "ffn_out"}, LORA_TARGETS
# assert STRIDE <= WINDOW <= cfg.max_seq_len and WINDOW % SF == 0, "[c-3] WINDOW: multiple of 4, <= 1536"
# assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] run setup + which linears exist in the adapted stages
# # -------------------------------------------------------------------------------------
# FIRST = STAGES.index(LORA_FROM)
# BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
# random.seed(SEED); torch.manual_seed(SEED)
# RUN_NAME = f"c3_lora_{LORA_FROM}_r{LORA_RANK}_lsi_{DOC_REPR}"
# OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
# SAVE = f"{OUT_DIR}/best_trainable.pt"
# POS_WEIGHT = make_pos_weight(POS_WEIGHT_SCHEME, POS_WEIGHT_CAP)

# def stage_blocks(name):
#     """EncoderBlocks inside one stage (the hourglass holds the 4 bottleneck blocks)."""
#     m = stage_module(lm, name)
#     return list(m.bottleneck) if name == "hourglass" else [] if name == "ln" else [m]

# def encoder_linears(stages, which):
#     """Yield (parent, attr, nn.Linear) for the chosen linears in every block of the given stages.
#     c-3 attention has ONE fused qkv_proj (560 -> 1680) and no biases; the FFN linears have biases."""
#     for name in stages:
#         for blk in stage_blocks(name):
#             for w in which:
#                 if w in ("qkv_proj", "out_proj"):
#                     parent, attr = blk.self_attn, w
#                 else:
#                     parent, attr = blk.ffn.net, "0" if w == "ffn_in" else "2"
#                 yield parent, attr, getattr(parent, attr)

# def wrap_linears(stages, which, make_wrapper):
#     todo = list(encoder_linears(stages, which))
#     for parent, attr, lin in todo:
#         assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
#         setattr(parent, attr, make_wrapper(lin))
#     return len(todo)

# for _name in ("qkv_proj", "out_proj", "ffn_in", "ffn_out"):
#     _, _, _lin = next(encoder_linears(["pre0"], (_name,)))
#     print(f"  {_name:10s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")
# print(f"adapted stages: {STAGES[FIRST:-1]} ({sum(len(stage_blocks(s)) for s in STAGES[FIRST:])} blocks) | "
#       f"frozen + cached: embeddings, {STAGES[:FIRST]}")


# # -------------------------------------------------------------------------------------
# # [L] LoRA layer — yours
# # -------------------------------------------------------------------------------------
# class LoRALinear(nn.Module):
#     """Wraps ONE pretrained nn.Linear of the c-3 encoder.
#     Constructed as   LoRALinear(base, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)   by wrap_linears below:
#         base    : the original nn.Linear (weight (out_features, in_features), bias may be None)
#         rank, alpha, dropout : your [H] values (alpha / dropout may be None if you don't use them)
#     forward(x)
#         x      : (B, T, in_features)
#         return : (B, T, out_features)   SAME shape as base(x): it goes straight back into the block
#     What the plumbing expects:
#         - the pretrained weights stay frozen (requires_grad False); only your new parameters train.
#           Everything with requires_grad True is optimised and saved to best_trainable.pt.
#         - keep self.base as an attribute.
#         - if your initialisation makes the layer start out equal to base, the identity check
#           below passes (diff ~0). If your design does not, set CHECK_IDENTITY = False.
#     """
#     def __init__(self, base: nn.Linear, rank: int, alpha=None, dropout=None):
#         super().__init__()
#         self.base = base
#         # TODO: freeze base, create your low-rank parameters and initialise them
#         raise NotImplementedError("write your LoRA layer")

#     def forward(self, x):
#         # TODO
#         raise NotImplementedError("write your LoRA forward")

# # [ANCILLARY] quick shape test on a dummy linear (no model involved)
# _t = LoRALinear(nn.Linear(FEAT_DIM, 3 * FEAT_DIM, bias=False), LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
# _y = _t(torch.randn(2, 8, FEAT_DIM))
# assert _y.shape == (2, 8, 3 * FEAT_DIM), f"LoRALinear output shape {tuple(_y.shape)}, expected (2, 8, {3 * FEAT_DIM})"
# assert not any(p.requires_grad for p in _t.base.parameters()), "freeze the base linear inside LoRALinear"
# print(f"LoRALinear trainable params on a {FEAT_DIM}->{3 * FEAT_DIM} layer:", sum(p.numel() for p in _t.parameters() if p.requires_grad))
# del _t, _y


# # -------------------------------------------------------------------------------------
# # [N] Head — yours (c-6 Phase 5 mean + max pool head, unchanged)
# # -------------------------------------------------------------------------------------
# class LSIHead(nn.Module):
#     def __init__(self, in_dim, n_labels):
#         super().__init__()
#         self.norm = nn.LayerNorm(2 * in_dim)
#         self.mlp = nn.Sequential(
#             nn.Linear(2 * in_dim, in_dim),
#             nn.GELU(),
#             nn.Dropout(0.1),
#             nn.Linear(in_dim, n_labels)
#         )

#     def forward(self, h):
#         mean_pool = h.mean(dim=1)
#         max_pool = h.max(dim=1).values
#         pooled = torch.cat([mean_pool, max_pool], dim=-1)
#         return self.mlp(self.norm(pooled))


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] freeze c-3, wrap the chosen linears, identity check, model
# # -------------------------------------------------------------------------------------
# CHECK_IDENTITY = True
# lm.eval()
# _ids = next(iter(train.values()))["ids"][:64]
# probe = torch.tensor([_ids + [PAD_ID] * (64 - len(_ids))], device=device)
# with torch.no_grad():
#     ref = encoder_forward(lm, probe).float()
# for p in lm.parameters():
#     p.requires_grad = False
# n_wrapped = wrap_linears(STAGES[FIRST:], LORA_TARGETS,
#                          lambda lin: LoRALinear(lin, LORA_RANK, LORA_ALPHA, LORA_DROPOUT).to(device))
# print(f"wrapped {n_wrapped} linears")
# if CHECK_IDENTITY:
#     with torch.no_grad():
#         diff = (encoder_forward(lm, probe).float() - ref).abs().max().item()
#     print(f"LoRA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
#     assert diff < 1e-3, "wrapped model differs from the original at init"
# assert not any(p.requires_grad for s in STAGES[:FIRST] for p in stage_module(lm, s).parameters()), \
#     "a stage below LORA_FROM has trainable weights: the cache would be wrong"

# head = LSIHead(FEAT_DIM, N_LABELS)
# model = PartialLSIModel(lm, head, FIRST).to(device)
# n_train_params, n_total_params = count_params(model)


# # -------------------------------------------------------------------------------------
# # [ANCILLARY] data, cache, checks, training, threshold tuning, scoring, submission
# # -------------------------------------------------------------------------------------
# train_ex = build_examples(train, **BUILD)
# val_ex = build_examples(dev, **BUILD)
# train_cache = FrozenCache(model, train_ex, BATCH_SIZE, CACHE_WHERE, CACHE_DIR, "train", CACHE_GPU_MAX_GB)
# val_cache = FrozenCache(model, val_ex, BATCH_SIZE, CACHE_WHERE, CACHE_DIR, "dev", CACHE_GPU_MAX_GB)

# model.eval()
# with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
#     for _b in make_batches(val_ex, 4, shuffle=False)[:3]:
#         _d = (model(to_tensors(_b, val_cache)).float() - model(to_tensors(_b)).float()).abs().max().item()
#         print(f"cached vs full path, {len(_b)} windows x {_b[0]['n']} tokens: max |logit diff| {_d:.2e}")
#         assert _d < 5e-2, "cache does not match the frozen stages"

# loss_fn = lambda logits, b: bce_loss(logits, b["labels"], POS_WEIGHT)
# if TIME_VS_FULL_FT:
#     t_step_part, t_step_full = step_time_vs_full_ft(model, train_ex, BATCH_SIZE, train_cache, loss_fn)

# log, best_res = train_model(
#     model, train_ex, val_ex, dev, loss_fn,
#     lr=LR, head_lr=HEAD_LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
#     lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD, min_labels=MIN_LABELS,
#     max_epoch_minutes=MAX_EPOCH_MINUTES, train_cache=train_cache, val_cache=val_cache)
# train_mem = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float("nan")

# model.load_state_dict(torch.load(SAVE), strict=False)
# dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG, val_cache)
# print(f"\n-- dev at the fixed THRESHOLD {THRESHOLD}")
# evaluate(decide(dev_probs, THRESHOLD, MIN_LABELS), dev)
# FINAL_THR = THRESHOLD if THRESH_MODE == "fixed" else tune_thresholds(dev_probs, dev, THRESH_MODE, THRESH_GRID, MIN_LABELS)
# dev_pred = decide(dev_probs, FINAL_THR, MIN_LABELS)
# print(f"\n-- dev with the final thresholds ({THRESH_MODE})")
# res = evaluate(dev_pred, dev)
# per_statute_report(res)
# error_cases(dev_pred, dev)

# summary = {"run": RUN_NAME, "technique": f"LoRA from {LORA_FROM}", "rank": LORA_RANK, "alpha": LORA_ALPHA,
#            "targets": list(LORA_TARGETS), "n_wrapped": n_wrapped, "build": BUILD,
#            "trainable_M": round(n_train_params / 1e6, 3), "total_M": round(n_total_params / 1e6, 2),
#            "cache_build_min": round((train_cache.build_s + val_cache.build_s) / 60, 2),
#            "cache_where": train_cache.where,
#            "train_min_per_epoch": round(float(np.mean([l["train_min"] for l in log])), 2),
#            "dev_min_per_epoch": round(float(np.mean([l["dev_min"] for l in log])), 2),
#            "step_s_lora": round(t_step_part, 3) if TIME_VS_FULL_FT else None,
#            "step_s_full_ft": round(t_step_full, 3) if TIME_VS_FULL_FT else None,
#            "dev_infer_s_full_path": round(time_inference(model, val_ex, BATCH_SIZE, AGG), 1),
#            "peak_train_gb": round(train_mem, 2), "thresh_mode": THRESH_MODE,
#            "dev": {k: round(res[k], 4) for k in ("f1", "f1_all", "micro_f1", "subset_acc")}}
# print(summary)
# json.dump({**summary, "log": log, "final_thr": np.asarray(FINAL_THR).tolist(),
#            "dev_probs": {k: v.round(4).tolist() for k, v in dev_probs.items()}},
#           open(f"{OUT_DIR}/dev_log.json", "w"))

# write_submission(model, BUILD, BATCH_SIZE, AGG, FINAL_THR, MIN_LABELS, best_path=SAVE, out_dir=OUT_DIR)
# for _c in (train_cache, val_cache):          # free the cache (GPU / RAM / file in CACHE_DIR)
#     if _c.where == "disk":
#         del _c.store; os.remove(_c.path)
# del train_cache, val_cache, _c
# if CUDA: torch.cuda.empty_cache()